# AgeLens - Task 05: features FaRL congeladas

Representacoes FP32 de 512 dimensoes, com L2 por imagem e preprocessamento oficial. Sem treinamento, ajuste de scaler/PCA, estimativa de idade ou alteracao dos splits. O cache privado e gerado pela CLI e somente consumido por este notebook.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/features-v1.json').exists())
assert Path(sys.prefix).resolve() == (ROOT / '.venv').resolve()
sys.path.insert(0, str(ROOT))
from agelens.cache_access import load_active_cache

directory, features, rows, stats, metadata = load_active_cache(ROOT)
verification = json.loads((ROOT / 'reports/task05/verification.json').read_text())
assert verification['completed'] and not verification['trained']
assert verification['cache_fingerprint'] == metadata['fingerprint']
print(json.dumps(stats, indent=2))

{
  "shape": [
    15000,
    512
  ],
  "dtype": "float32",
  "finite": true,
  "min_row_norm": 0.9999998211860657,
  "max_row_norm": 1.0000001192092896,
  "min_column_std": 0.010070926156589288
}


## Correspondencia por particao

Os indices sao derivados dos manifests congelados. Os tres folds sao exclusivos do treino; -1 indica que a particao nao participa da validacao cruzada. Nenhum dado e redistribuido.

In [2]:
summary = []
for name in ['train','val_select','val_calibration','test']:
    indices = np.load(directory / f'indices_{name}.npy', allow_pickle=False)
    assert rows.iloc[indices]['split'].eq(name).all()
    summary.append({'particao':name, 'imagens':len(indices), 'dimensoes':features.shape[1], 'folds':sorted(map(int, rows.iloc[indices].cv_fold.unique()))})
display(pd.DataFrame(summary))

,particao,imagens,dimensoes,folds
0,train,9001,512,"[0, 1, 2]"
1,val_select,1500,512,[-1]
2,val_calibration,1499,512,[-1]
3,test,3000,512,[-1]


## Verificacao independente

O comprovante registra integridade de todos os arquivos/imagens selecionadas e recomputacao de quatro imagens por particao apos recarregar o checkpoint. Diferencas entre inferencia individual e em lote sao avaliadas com tolerancia FP32 explicita, nao igualdade artificial.

In [3]:
keys = ['samples','feature_shape','dtype','finite','l2_normalized','all_selected_image_hashes_verified','all_cache_file_hashes_verified','independent_recomputed_rows','recomputation_max_absolute_error','recomputation_tolerance','single_image_repeats_bitwise_equal','encoder_state_unchanged','all_parameters_frozen','gradients_absent','unit_tests_passed','trained']
print(json.dumps({key:verification[key] for key in keys}, indent=2))

{
  "samples": 15000,
  "feature_shape": [
    15000,
    512
  ],
  "dtype": "float32",
  "finite": true,
  "l2_normalized": true,
  "all_selected_image_hashes_verified": true,
  "all_cache_file_hashes_verified": true,
  "independent_recomputed_rows": 16,
  "recomputation_max_absolute_error": 2.4586915969848633e-07,
  "recomputation_tolerance": {
    "atol": 2e-06,
    "rtol": 1e-05
  },
  "single_image_repeats_bitwise_equal": true,
  "encoder_state_unchanged": true,
  "all_parameters_frozen": true,
  "gradients_absent": true,
  "unit_tests_passed": 50,
  "trained": false
}


## Limites e proxima etapa

Features faciais permanecem fora do Git. O cache nao comprova idade real, identidade ou desempenho de classificacao. O risco de identidade repetida entre fotos diferentes continua sendo uma limitacao dos metadados. A task 06 preparara MLflow; os modelos e suas metricas serao trabalhados somente nas etapas de treinamento autorizadas.